# Discovery dataset - Resting MRS-voxel comparisons

In [5]:
from pathlib import Path
import sys
base_path = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "CODE").is_dir() and (p / "DATA").is_dir()
)

code_dir = str(Path(base_path) / "CODE")
if code_dir not in sys.path:
    sys.path.insert(0, code_dir)

In [6]:
import os
import numpy as np
import pandas as pd
from itertools import combinations
from scipy import stats
from statsmodels.stats.multitest import multipletests

cleaned_df_path = f"{base_path}/DATA/discovery_dataset/cleaned_MRS_df.csv"

out_dir = os.path.join(base_path, "RESULTS/discovery_dataset/statistics/EIB_repeated_measures")
os.makedirs(out_dir, exist_ok=True)

alpha = 0.05

voxel_order = [
    "anteriorcingulate",
    "occipital",
    "rightauditory",
    "smleft",
]

eib_col = "GlxGABA_AlphaTissCorr"

In [7]:
from utils.mrs import parse_real_numeric_array, normality_test

In [8]:
# Load data

cleaned_df = pd.read_csv(cleaned_df_path)
df = cleaned_df.copy()

if "Status" in df.columns:
    df = df[df["Status"].astype(str).str.lower() == "ok"].copy()

if "Overall_eligible" in df.columns:
    overall = df["Overall_eligible"].astype(str).str.lower().isin(["true", "1", "t", "yes"])
    df = df[overall].copy()

df = df[df["Voxel"].isin(voxel_order)].copy()
df["EIB"] = parse_real_numeric_array(df[eib_col].values)
df = df[["Subject", "Voxel", "EIB"]].dropna().copy()
df["Voxel"] = pd.Categorical(df["Voxel"], categories=voxel_order, ordered=True)
df.to_csv(os.path.join(out_dir, "EIB_long.csv"), index=False)


# 2. Wide format for repeated-measures analysis
# One row per subject, one column per voxel
wide = df.pivot_table(
    index="Subject",
    columns="Voxel",
    values="EIB",
    aggfunc="mean"
)

# Keep only complete cases across all four voxels
wide_complete = wide[voxel_order].dropna().copy()
wide_complete.to_csv(os.path.join(out_dir, "EIB_wide_complete.csv"))

In [9]:
# Normality check per voxel
normality_rows = []

for vx in voxel_order:
    x = wide_complete[vx].values
    stat, p = normality_test(x)

    normality_rows.append({
        "Voxel": vx,
        "Test": "Shapiro",
        "Statistic": stat,
        "P": p,
        "Pass_p_gt_0.05": p > alpha if np.isfinite(p) else False,
        "N": len(x)
    })

normality_df = pd.DataFrame(normality_rows)

print("\nNormality per voxel:")
print(normality_df)


Normality per voxel:
               Voxel     Test  Statistic         P  Pass_p_gt_0.05    N
0  anteriorcingulate  Shapiro   0.982388  0.085359            True  132
1          occipital  Shapiro   0.949133  0.000086           False  132
2      rightauditory  Shapiro   0.976468  0.021568           False  132
3             smleft  Shapiro   0.967987  0.003309           False  132


In [10]:
omnibus_name = "Friedman test"

friedman_stat, friedman_p = stats.friedmanchisquare(
    *[wide_complete[vx].values for vx in voxel_order]
)

# Kendall's W effect size for Friedman
k = len(voxel_order)
n = wide_complete.shape[0]
kendalls_w = friedman_stat / (n * (k - 1))

omnibus_df = pd.DataFrame([{
    "Chosen_test": omnibus_name,
    "Statistic": friedman_stat,
    "P": friedman_p,
    "N_subjects": n,
    "N_conditions": k,
    "Kendalls_W": kendalls_w
}])

omnibus_df.to_csv(os.path.join(out_dir, "omnibus_repeated_measures_test.csv"), index=False)

print("\nChosen omnibus test:")
print(omnibus_df)


Chosen omnibus test:
     Chosen_test   Statistic             P  N_subjects  N_conditions  \
0  Friedman test  287.645455  4.693037e-62         132             4   

   Kendalls_W  
0    0.726377  


In [11]:
from utils.mrs import wilcoxon_effect_size, effect_size_magnitude

# Pairwise repeated-measures tests

pairwise_rows = []

for v1, v2 in combinations(voxel_order, 2):
    x = wide_complete[v1].values
    y = wide_complete[v2].values

    differences = x - y

    (
        stat,
        p_raw,
        z_value,
        effect_size_r,
        n_effective
    ) = wilcoxon_effect_size(x, y)

    median_difference = np.median(differences)

    if median_difference > 0:
        direction = f"{v1} > {v2}"
    elif median_difference < 0:
        direction = f"{v1} < {v2}"
    else:
        direction = "equal median difference"

    pairwise_rows.append({
        "Voxel1": v1,
        "Voxel2": v2,
        "Test": "Wilcoxon signed-rank",

        # Test statistics
        "Wilcoxon_statistic": stat,
        "Wilcoxon_Z": z_value,
        "P_raw": p_raw,

        # Descriptive statistics
        "Mean_Voxel1": np.mean(x),
        "Mean_Voxel2": np.mean(y),
        "Median_Voxel1": np.median(x),
        "Median_Voxel2": np.median(y),

        "MeanDiff_Voxel1_minus_Voxel2": np.mean(differences),
        "MedianDiff_Voxel1_minus_Voxel2": median_difference,
        "Direction": direction,

        # Effect size
        "Wilcoxon_effect_size_r": effect_size_r,
        "Effect_size_magnitude": effect_size_magnitude(effect_size_r),

        # Sample sizes
        "N_total_pairs": len(differences),
        "N_nonzero_differences": n_effective
    })

pairwise_df = pd.DataFrame(pairwise_rows)

# Benjamini-Hochberg correction
reject, p_fdr, _, _ = multipletests(
    pairwise_df["P_raw"].values,
    method="fdr_bh"
)

pairwise_df["P_FDR_BH"] = p_fdr
pairwise_df["Reject_FDR_0.05"] = reject

pairwise_df = pairwise_df.sort_values("P_FDR_BH")

pairwise_df.to_csv(
    os.path.join(
        out_dir,
        "pairwise_repeated_measures_FDR.csv"
    ),
    index=False
)

print("\nPairwise repeated-measures comparisons:")

display(
    pairwise_df[
        [
            "Voxel1",
            "Voxel2",
            "MedianDiff_Voxel1_minus_Voxel2",
            "Direction",
            "Wilcoxon_statistic",
            "Wilcoxon_Z",
            "Wilcoxon_effect_size_r",
            "Effect_size_magnitude",
            "P_FDR_BH",
            "Reject_FDR_0.05",
            "N_total_pairs",
            "N_nonzero_differences"
        ]
    ]
)


Pairwise repeated-measures comparisons:


,Voxel1,Voxel2,MedianDiff_Voxel1_minus_Voxel2,Direction,Wilcoxon_statistic,Wilcoxon_Z,Wilcoxon_effect_size_r,Effect_size_magnitude,P_FDR_BH,Reject_FDR_0.05,N_total_pairs,N_nonzero_differences
0,anteriorcingulate,occipital,0.789226,anteriorcingulate > occipital,11.0,-9.943649,0.865484,large,8.063876e-23,True,132,132
2,anteriorcingulate,smleft,0.801169,anteriorcingulate > smleft,0.0,-9.968633,0.867658,large,8.063876e-23,True,132,132
3,occipital,rightauditory,-0.470347,occipital < rightauditory,380.0,-9.105548,0.792536,large,1.716638e-19,True,132,132
5,rightauditory,smleft,0.511225,rightauditory > smleft,460.0,-8.923846,0.776721,large,6.755550e-19,True,132,132
1,anteriorcingulate,rightauditory,0.352868,anteriorcingulate > rightauditory,546.0,-8.728516,0.759720,large,3.096426e-18,True,132,132
4,occipital,smleft,0.007138,occipital > smleft,4022.0,-0.833559,0.072552,negligible,4.045298e-01,False,132,132
